# 🟤 Locksley Resources Limited (ASX: LKY | OTCQX: LKYRF)
## Comprehensive Copper-Gold & Critical Minerals Investment Analysis
### Built with the Copper Miner Analyzer Skill | February 2026

---

> **⚠️ IMPORTANT CONTEXT:** Locksley Resources is an **exploration-stage company** with no current production.
> The primary analytical framework used here is **in-situ resource valuation**, **probability-weighted
> discovery value**, and a **prospect-to-mine DCF** with appropriate development risk discounts.
> This differs from producer-level analysis applied to operating miners.

---

**Key Company Facts (as of February 2026)**
| Item | Detail |
|------|--------|
| Ticker | ASX: LKY / OTCQX: LKYRF / FRA: X5L |
| Exchange | Australian Securities Exchange (ASX) |
| Headquarters | Perth, Western Australia |
| Founded | 2018 |
| Stage | Exploration — pre-PEA |
| Current Price | ~A$0.335 |
| Market Cap | ~A$96.6M |
| Shares Outstanding | ~288M (estimated post-placements) |
| Cash (post Aug 2025 raise) | ~A$6M+ |
| Key Projects | Tottenham Cu-Au (NSW) + Mojave Sb/REE (California) |
| Copper Resource (JORC Inferred) | 9.86Mt @ 0.72% Cu, 0.22g/t Au, 2g/t Ag |

---

**Analyst Note:** *The stock went from A$0.014 (Jan 2025) to an ATH of A$0.69 (Sep 2025) — a 4,800% move — 
driven by the pivot to US critical minerals (antimony + REE). The copper project (Tottenham) provides 
fundamental resource base value; the Mojave project provides the speculative re-rating optionality.*

## Section 0: Configuration — Edit These Parameters

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
#  LOCKSLEY RESOURCES LIMITED (LKY.ASX) — COPPER MINER ANALYSIS
#  Built by Claude · February 2026 · Edit variables below to customise
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── COMPANY ──────────────────────────────────────────────────────────────────
COMPANY_NAME        = "Locksley Resources Limited"
TICKER              = "LKY.ASX"
SHARES_OUT_M        = 288.0          # million shares outstanding (ESTIMATED - post placements)
CURRENT_PRICE_AUD   = 0.335          # AUD/share (Feb 2026)
AUD_USD             = 0.630          # AUD/USD exchange rate
CURRENT_PRICE_USD   = CURRENT_PRICE_AUD * AUD_USD  # ~USD 0.211/share
NET_DEBT_USD_M      = -3.8           # Net cash position (negative = net cash); ~A$6M cash, minimal debt → ~USD 3.8M net cash
MARKET_CAP_AUD_M    = SHARES_OUT_M * CURRENT_PRICE_AUD  # ~A$96.6M
MARKET_CAP_USD_M    = MARKET_CAP_AUD_M * AUD_USD

# ── COPPER PRICE ASSUMPTIONS (USD/lb) ────────────────────────────────────────
LIVE_COPPER_PRICE_LB  = 5.90   # COMEX Feb 2026 — near ATH territory
COPPER_PRICE_BEAR_LB  = 3.75   # Bear: demand destruction / recession
COPPER_PRICE_BASE_LB  = 5.00   # Base: normalised post-tariff supercycle
COPPER_PRICE_BULL_LB  = 6.50   # Bull: structural deficit deepens, AI + grid demand surges
COPPER_ATH_LB         = 6.61   # All-time high: Jan 29, 2026

# ── BY-PRODUCT PRICE ASSUMPTIONS ─────────────────────────────────────────────
GOLD_PRICE_OZ     = 2900    # USD/oz — gold near record highs Feb 2026
SILVER_PRICE_OZ   = 32      # USD/oz
MOLY_PRICE_LB     = 22      # USD/lb
ANTIMONY_PRICE_LB = 25      # USD/lb — geopolitical premium, China export controls

# ── CONVERSION ────────────────────────────────────────────────────────────────
LBS_PER_TONNE     = 2204.62  # 1 metric tonne = 2204.62 lbs
KT_TO_T           = 1000     # 1 ktonne = 1000 tonnes

# ── DISCOUNT RATES ────────────────────────────────────────────────────────────
WACC              = 0.12     # 12% — higher for junior explorer (jurisdiction, stage risk)
TERMINAL_GROWTH   = 0.02
EXPLORATION_RISK  = 0.30     # probability of successful resource definition → mine (30%)
DEVELOPMENT_RISK  = 0.50     # probability of development given resource (50%)
COMBINED_SUCCESS  = EXPLORATION_RISK * DEVELOPMENT_RISK  # 15% risk-adjusted probability

# ── DCF HORIZON ───────────────────────────────────────────────────────────────
START_YEAR        = 2030     # Earliest possible first production given current stage
FORECAST_YEARS    = 15       # Mine life
PRE_PRODUCTION_YEARS = 5     # Years from now until production (2026-2030)

# ── TOTTENHAM PROJECT — RESOURCE (JORC INFERRED, 2023) ────────────────────────
TOTTENHAM_RESOURCE_MT   = 9.86    # million tonnes ore
TOTTENHAM_GRADE_CU_PCT  = 0.72    # % Cu
TOTTENHAM_GRADE_AU_GT   = 0.22    # g/t Au
TOTTENHAM_GRADE_AG_GT   = 2.0     # g/t Ag
TOTTENHAM_CU_CONTAINED_KT = TOTTENHAM_RESOURCE_MT * TOTTENHAM_GRADE_CU_PCT / 100 * 1000  # ktonne Cu
TOTTENHAM_AU_CONTAINED_KOZ = TOTTENHAM_RESOURCE_MT * TOTTENHAM_GRADE_AU_GT / 31.1035 * 1000  # koz Au
TOTTENHAM_OWNERSHIP_PCT = 100     # 100% owned

# ── EXPLORATION UPSIDE SCENARIO ───────────────────────────────────────────────
TOTTENHAM_EXPANDED_MT   = 50.0    # Bull case resource expansion (50km corridor)
TOTTENHAM_EXPANDED_GRADE= 0.60    # Typical for expanded VMS system

# ── CORPORATE G&A (USD millions per year) ─────────────────────────────────────
GA_BASE_USD_M     = 4.0      # Low for explorer (A$~6M/yr burn rate)

print("=" * 65)
print(f"  {COMPANY_NAME} — Configuration Loaded")
print("=" * 65)
print(f"  Ticker:          {TICKER}")
print(f"  Current Price:   A${CURRENT_PRICE_AUD:.3f} (USD${CURRENT_PRICE_USD:.3f})")
print(f"  Market Cap:      A${MARKET_CAP_AUD_M:.1f}M (USD${MARKET_CAP_USD_M:.1f}M)")
print(f"  Cu Resource:     {TOTTENHAM_CU_CONTAINED_KT:.1f}kt Cu in-situ (JORC Inferred)")
print(f"  Au Resource:     {TOTTENHAM_AU_CONTAINED_KOZ:.0f}koz Au in-situ")
print(f"  Live Copper:     USD${LIVE_COPPER_PRICE_LB:.2f}/lb (COMEX)")
print(f"  WACC:            {WACC*100:.0f}% (exploration-stage premium)")
print(f"  Combined P(success): {COMBINED_SUCCESS*100:.0f}% (exploration × development)")
print("=" * 65)
print("  ⚡ Edit variables above to update all outputs")
print("=" * 65)


  Locksley Resources Limited — Configuration Loaded
  Ticker:          LKY.ASX
  Current Price:   A$0.335 (USD$0.211)
  Market Cap:      A$96.5M (USD$60.8M)
  Cu Resource:     71.0kt Cu in-situ (JORC Inferred)
  Au Resource:     70koz Au in-situ
  Live Copper:     USD$5.90/lb (COMEX)
  WACC:            12% (exploration-stage premium)
  Combined P(success): 15% (exploration × development)
  ⚡ Edit variables above to update all outputs


## Section 1: Macro Copper Demand — The Supercycle Thesis

### Why Copper Is at All-Time Highs in 2026

Copper hit a **record $6.61/lb** on January 29, 2026 — an 11% intraday surge — driven by:

1. **AI Infrastructure Buildout**: Data centres consume ~20,000 kg of copper per MW. The AI 
   infrastructure wave (Stargate $500B, hyperscaler capex explosion) is adding an entirely new 
   demand vector that didn't exist in prior commodity cycles.
   
2. **Trump Tariff Front-Running**: Section 232 investigation into copper imports prompted 
   aggressive US stockpiling, creating a structural COMEX premium over LME.
   
3. **Energy Transition Supercycle**: EVs (83kg Cu vs 23kg ICE), offshore wind (12,000 kg/MW), 
   solar (4,500 kg/MW), and grid modernisation are compounding demand with no cyclical offset.

4. **Mine Supply Stagnation**: Grade decline at major porphyry mines (Escondida, Grasberg), 
   permitting delays, water scarcity in Chile, and resource nationalism are capping supply growth.

5. **Chinese Stimulus**: Recovery in Chinese construction and manufacturing adding base demand.

**Goldman Sachs estimates the copper market faced a 600,000t surplus in 2025 despite record 
prices** — yet prices broke records. This reflects market forward-pricing of structural deficits 
expected from 2027-2030 as mine supply peaks and transition demand accelerates.

> **LKY Relevance:** Locksley's Tottenham VHMS deposit would be a direct beneficiary of elevated 
> copper prices. A higher copper price increases the economics of marginal deposits significantly.


In [2]:
# ── Macro Copper Demand Model (ktpa refined copper) ──────────────────────────
years = list(range(2024, 2036))

demand_data = {
    "Grid & Transmission":     [7200, 7500, 7900, 8400, 8900, 9500, 10100, 10800, 11500, 12300, 13100, 14000],
    "Construction":            [6800, 6900, 7000, 7100, 7200, 7300,  7400,  7500,  7600,  7700,  7800,  7900],
    "EVs & Charging":          [1200, 1600, 2100, 2700, 3400, 4200,  5100,  6000,  7000,  8000,  9000, 10000],
    "Renewables (Wind+Solar)": [1800, 2100, 2500, 2900, 3400, 3900,  4500,  5100,  5800,  6500,  7200,  8000],
    "Industrial Machinery":    [4500, 4600, 4700, 4800, 4900, 5000,  5100,  5200,  5300,  5400,  5500,  5600],
    "Consumer Electronics":    [2800, 2850, 2900, 2950, 3000, 3050,  3100,  3150,  3200,  3250,  3300,  3350],
    "Data Centres & AI":       [ 400,  550,  750, 1000, 1300, 1650,  2050,  2500,  3000,  3550,  4150,  4800],
    "Other":                   [1800, 1850, 1900, 1950, 2000, 2050,  2100,  2150,  2200,  2250,  2300,  2350],
}

df_demand = pd.DataFrame(demand_data, index=years)

fig = go.Figure()
colors = ['#1565C0','#795548','#4CAF50','#FFC107','#9E9E9E','#9C27B0','#FF5722','#607D8B']
for i, col in enumerate(df_demand.columns):
    fig.add_trace(go.Scatter(
        x=years, y=df_demand[col], name=col,
        stackgroup='one', fillcolor=colors[i % len(colors)],
        line=dict(color=colors[i % len(colors)])
    ))

fig.update_layout(
    title="<b>Global Copper Demand by Sector 2024–2035 (ktpa)</b><br><sub>AI/Data Centre demand the fastest-growing new vector | Source: IEA, CRU estimates (Claude model)</sub>",
    xaxis_title="Year", yaxis_title="ktpa Refined Copper",
    hovermode="x unified", template="plotly_white",
    legend=dict(orientation="h", y=-0.35),
    height=500
)
fig.show()

total_demand_2024 = df_demand.loc[2024].sum()
total_demand_2035 = df_demand.loc[2035].sum()
growth_pct = (total_demand_2035 / total_demand_2024 - 1) * 100
print(f"Total demand 2024: {total_demand_2024:,.0f} ktpa")
print(f"Total demand 2035: {total_demand_2035:,.0f} ktpa")
print(f"Total growth:      +{growth_pct:.1f}% over 11 years")
print(f"AI/Data Centre grows from {df_demand.loc[2024,'Data Centres & AI']} to {df_demand.loc[2035,'Data Centres & AI']} ktpa ({(df_demand.loc[2035,'Data Centres & AI']/df_demand.loc[2024,'Data Centres & AI']-1)*100:.0f}% growth)")


Total demand 2024: 26,500 ktpa
Total demand 2035: 56,000 ktpa
Total growth:      +111.3% over 11 years
AI/Data Centre grows from 400 to 4800 ktpa (1100% growth)


In [3]:
# ── Supply/Demand Balance ─────────────────────────────────────────────────────
mine_supply    = [22500,23000,23400,23700,24000,24200,24400,24600,24800,25000,25200,25400]
secondary_supply=[5500, 5700, 5900, 6100, 6300, 6500, 6700, 6900, 7100, 7300, 7500, 7700]
total_supply   = [m + s for m, s in zip(mine_supply, secondary_supply)]
total_demand   = df_demand.sum(axis=1).values
deficit        = total_demand - np.array(total_supply)

fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(go.Bar(x=years, y=total_supply, name="Total Supply (Mine + Scrap)",
                      marker_color='steelblue'), secondary_y=False)
fig2.add_trace(go.Bar(x=years, y=total_demand, name="Total Demand",
                      marker_color='coral'), secondary_y=False)
fig2.add_trace(go.Scatter(x=years, y=deficit, name="Deficit(+) / Surplus(-)",
                          line=dict(color='red', width=3, dash='dot'),
                          mode='lines+markers'), secondary_y=True)
fig2.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3, secondary_y=True)

fig2.update_layout(
    title="<b>Copper Supply vs Demand Balance 2024–2035 (ktpa)</b><br><sub>Structural deficit emerges 2027+ as mine supply peaks</sub>",
    barmode='group', template="plotly_white", height=450
)
fig2.update_yaxes(title_text="ktpa", secondary_y=False)
fig2.update_yaxes(title_text="Deficit/Surplus (ktpa)", secondary_y=True)
fig2.show()

# Deficit analysis
first_deficit_yr = [years[i] for i, d in enumerate(deficit) if d > 0]
print(f"First structural deficit year: {first_deficit_yr[0] if first_deficit_yr else 'Beyond 2035'}")
print(f"Peak deficit (2035): {deficit[-1]:,.0f} ktpa")


First structural deficit year: 2026
Peak deficit (2035): 22,900 ktpa


In [4]:
# ── Copper Intensity in Energy Transition ────────────────────────────────────
tech_data = {
    'Technology': ['ICE Vehicle', 'Battery EV', 'Onshore Wind (per MW)',
                   'Offshore Wind (per MW)', 'Solar PV (per MW)',
                   'Data Centre (per MW)', 'Grid Battery (per MWh)'],
    'Copper (kg)': [23, 83, 4000, 12000, 4500, 20000, 1500],
    'Category': ['Transport','Transport','Renewables','Renewables','Renewables','Digital','Storage']
}
df_int = pd.DataFrame(tech_data)
color_map = {'Transport':'#2196F3','Renewables':'#4CAF50','Digital':'#FF5722','Storage':'#9C27B0'}

fig3 = px.bar(df_int, x='Technology', y='Copper (kg)', color='Category',
              color_discrete_map=color_map,
              title="<b>Copper Intensity by Technology</b><br><sub>Data centres most copper-intensive per MW — the AI demand vector</sub>",
              text='Copper (kg)')
fig3.update_traces(texttemplate='%{text:,.0f} kg', textposition='outside')
fig3.update_layout(template='plotly_white', yaxis_title='Copper Content (kg)',
                   showlegend=True, height=450)
fig3.show()

print("Key insight: A single 100MW data centre requires ~2,000 TONNES of copper")
print(f"That's equivalent to {2000/TOTTENHAM_CU_CONTAINED_KT*100:.1f}% of Tottenham's current Cu resource!")


Key insight: A single 100MW data centre requires ~2,000 TONNES of copper
That's equivalent to 2817.2% of Tottenham's current Cu resource!


## Section 2: Company Overview

### Investment Thesis — A Dual-Optionality Play

Locksley Resources is a compelling and unusual story for 2026: it offers **rare dual exposure** to 
two of the most geopolitically-charged commodity themes:

1. **🟤 Copper/Gold (Australia)** — The Tottenham Project in NSW provides a **JORC-compliant inferred
   resource** in a proven mining belt, offering leverage to the global copper supercycle.

2. **⚫ Antimony/REE (USA)** — The Mojave Project in California sits 1.4km from MP Materials' 
   Mountain Pass Mine (the only producing REE mine in North America) and hosts **one of the 
   highest-grade undeveloped antimony occurrences in the United States**. With China controlling 
   >90% of antimony supply and implementing export controls in 2024, this is a genuine strategic 
   asset for US defence and energy security.

### The Stock's Extraordinary 2025 Journey
- Jan 2025: A$0.014 (lowest price)
- Sep 2025: A$0.69 (ATH — +4,828% in 9 months)
- Feb 2026: ~A$0.335 (pullback from ATH but still 2,300%+ up from lows)
- Catalyst: Antimony narrative + US critical minerals policy + rice University R&D deal + 
  EXIM Bank Letter of Interest + DoD/DoE alignment


In [5]:
# ── Company Overview Stats ────────────────────────────────────────────────────
print("=" * 65)
print(f"  {COMPANY_NAME}")
print("=" * 65)
print(f"  ASX Ticker:          LKY     | OTCQX: LKYRF | FRA: X5L")
print(f"  Incorporated:        2018    | Perth, WA, Australia")
print(f"  Stage:               EXPLORATION (pre-PEA, no production)")
print()
print(f"  ── Market Data ───────────────────────────────────────────")
print(f"  Share Price (AUD):   A${CURRENT_PRICE_AUD:.3f}")
print(f"  Share Price (USD):   USD${CURRENT_PRICE_USD:.3f}")
print(f"  Shares Outstanding:  ~{SHARES_OUT_M:.0f}M")
print(f"  Market Cap (AUD):    A${MARKET_CAP_AUD_M:.1f}M")
print(f"  Market Cap (USD):    USD${MARKET_CAP_USD_M:.1f}M")
print(f"  52-Week Range:       A$0.014 – A$0.690")
print(f"  Cash (est.):         ~A$6M+ (post Aug 2025 placement)")
print()
print(f"  ── Resources (JORC 2012 Inferred, Tottenham) ────────────")
print(f"  Ore Tonnes:          {TOTTENHAM_RESOURCE_MT:.2f}Mt @ {TOTTENHAM_GRADE_CU_PCT:.2f}% Cu cut-off 0.3%")
print(f"  Copper Contained:    {TOTTENHAM_CU_CONTAINED_KT:.1f}kt Cu ({TOTTENHAM_CU_CONTAINED_KT*LBS_PER_TONNE/1e6:.2f}B lbs)")
print(f"  Gold Contained:      ~{TOTTENHAM_AU_CONTAINED_KOZ:.0f}koz Au")
print(f"  Silver Contained:    ~511koz Ag")
print(f"  Grade (Cu):          {TOTTENHAM_GRADE_CU_PCT:.2f}% (above typical VMS cut-off, high-grade veins to 10.5%)")
print()
print(f"  ── Implied Valuations ────────────────────────────────────")
in_situ_value_base = TOTTENHAM_CU_CONTAINED_KT * 1000 * LBS_PER_TONNE * COPPER_PRICE_BASE_LB / 1e6
in_situ_value_live = TOTTENHAM_CU_CONTAINED_KT * 1000 * LBS_PER_TONNE * LIVE_COPPER_PRICE_LB / 1e6
ev_per_cu_tonne = MARKET_CAP_USD_M / TOTTENHAM_CU_CONTAINED_KT
print(f"  In-Situ Cu Value (Base ${COPPER_PRICE_BASE_LB}/lb):  USD${in_situ_value_base:,.1f}M")
print(f"  In-Situ Cu Value (Live ${LIVE_COPPER_PRICE_LB}/lb):  USD${in_situ_value_live:,.1f}M")
print(f"  Market Cap vs In-Situ (Base):     {MARKET_CAP_USD_M/in_situ_value_base*100:.1f}% (typical explorer: 2-8%)")
print(f"  EV per tonne Cu in-situ:          USD${ev_per_cu_tonne:,.0f}/t")
print(f"  (Typical explorer range: USD$30-150/t for inferred resource)")
print("=" * 65)


  Locksley Resources Limited
  ASX Ticker:          LKY     | OTCQX: LKYRF | FRA: X5L
  Incorporated:        2018    | Perth, WA, Australia
  Stage:               EXPLORATION (pre-PEA, no production)

  ── Market Data ───────────────────────────────────────────
  Share Price (AUD):   A$0.335
  Share Price (USD):   USD$0.211
  Shares Outstanding:  ~288M
  Market Cap (AUD):    A$96.5M
  Market Cap (USD):    USD$60.8M
  52-Week Range:       A$0.014 – A$0.690
  Cash (est.):         ~A$6M+ (post Aug 2025 placement)

  ── Resources (JORC 2012 Inferred, Tottenham) ────────────
  Ore Tonnes:          9.86Mt @ 0.72% Cu cut-off 0.3%
  Copper Contained:    71.0kt Cu (0.16B lbs)
  Gold Contained:      ~70koz Au
  Silver Contained:    ~511koz Ag
  Grade (Cu):          0.72% (above typical VMS cut-off, high-grade veins to 10.5%)

  ── Implied Valuations ────────────────────────────────────
  In-Situ Cu Value (Base $5.0/lb):  USD$782.6M
  In-Situ Cu Value (Live $5.9/lb):  USD$923.4M
  Market Cap vs I

## Section 3: Project-by-Project Breakdown

### Project Portfolio Overview

Locksley holds two primary projects at very different stages:

| Project | Location | Primary Metal | Status | Resource |
|---------|----------|--------------|--------|----------|
| Tottenham | NSW, Australia | Cu, Au, Ag | JORC Inferred MRE | 9.86Mt @ 0.72% Cu |
| Mojave (Desert Antimony Mine) | California, USA | Sb, Ag | Pre-resource, drilling 2025 | Historical only |
| Mojave (El Campo REE) | California, USA | REE (NdPr) | Pre-resource, drilling 2025 | Rock chips 12.1% TREO |

### Tottenham Copper-Gold Project (FLAGSHIP COPPER ASSET)

**Location:** Cobar-Girilambone District, Central NSW — same belt as Aeris Resources' Tritton Mine  
**Type:** Volcanic Hosted Massive Sulphide (VHMS) — same metallogenic system as Hellyer, Rosebery  
**Land Package:** 470 km² across 4 exploration licences  
**Strike Length:** 50km VHMS copper corridor (highly under-drilled)  
**Key Deposits:** Carolina Deposit + Mount Royal–Orange Plains Deposits  

**High-Grade Intercepts (best results):**
- 20m @ 3.53% Cu, 0.1g/t Au (TPRC043), including 12m @ **5.64% Cu**
- 4.39m @ **10.5% Cu**, 1.8g/t Au (TMD002)  
- 19m @ 0.87% Cu from 32m (TPRC057)

**Exploration Upside:** DHEM surveys identified strong off-hole conductors → multiple untested targets on 50km corridor  
**Next Catalysts:** Updated resource estimate Q4 2025–Q1 2026; infill + extension drilling Q2–Q3 2025

### Mojave Critical Minerals Project (HIGH-RISK / HIGH-REWARD OPTIONALITY)

**Desert Antimony Mine (DAM):**
- Surface assays: up to **46% Sb**, 1,022 g/t Ag
- One of the highest-grade undeveloped antimony occurrences in the US
- Historical mine with existing adits/stopes (reduces drilling cost/risk)
- Maiden RC drilling approved by BLM, underway Q3-Q4 2025

**El Campo REE Target:**
- Rock chips: **12.1% TREO**, 3.19% NdPr (highest-value rare earths)
- 860m mineralised horizon, drill-ready, 1.4km from Mountain Pass Mine
- Strategic: NdPr used in EV motors, wind turbines, defence magnets

**Strategic Partnerships:**
- Rice University R&D Agreement (hydrometallurgical antimony extraction + battery applications)
- GreenMet (Washington DC) — government funding access (DoD, DoE, IRA programs)
- EXIM Bank Letter of Interest received
- Tribeca Capital as strategic advisor


In [6]:
# ── Project Map Visual ───────────────────────────────────────────────────────
projects = {
    "Tottenham Cu-Au NSW": {
        "type": "vms",
        "status": "Inferred MRE — expanding",
        "resource_mt": TOTTENHAM_RESOURCE_MT,
        "grade_cu_pct": TOTTENHAM_GRADE_CU_PCT,
        "contained_cu_kt": TOTTENHAM_CU_CONTAINED_KT,
        "contained_au_koz": TOTTENHAM_AU_CONTAINED_KOZ,
        "ownership_pct": 100,
        "target_production_ktpa_cu": 12.0,     # ESTIMATED — small VMS starter mine
        "first_production_year": 2031,         # ESTIMATED
        "mine_life_years": 15,                 # ESTIMATED
        "c1_cash_cost_lb": 1.80,              # ESTIMATED — VMS typically low C1 pre-credits
        "aisc_lb": 2.50,                      # ESTIMATED
        "capex_total_usdm": 150,              # ESTIMATED DFS/construction capex
        "capex_sustaining_annual_usdm": 8,    # ESTIMATED
        "royalty_pct": 2.5,                   # NSW state royalty
        "tax_rate": 0.30,                     # Australian corporate tax
        "country": "Australia",
        "gold_koz_yr": 8.0,                   # ESTIMATED from grade
        "silver_koz_yr": 75.0,               # ESTIMATED from grade
        "moly_mlbs_yr": 0,
    },
    "Mojave Desert Antimony (DAM)": {
        "type": "vms",
        "status": "Pre-resource — drilling 2025",
        "resource_mt": 0,
        "grade_cu_pct": 0,
        "contained_cu_kt": 0,
        "contained_au_koz": 0,
        "ownership_pct": 100,
        "target_production_ktpa_cu": 0,
        "first_production_year": 2029,
        "mine_life_years": 10,
        "c1_cash_cost_lb": 0,
        "aisc_lb": 0,
        "capex_total_usdm": 50,
        "capex_sustaining_annual_usdm": 5,
        "royalty_pct": 3.0,
        "tax_rate": 0.27,
        "country": "USA",
        "gold_koz_yr": 0,
        "silver_koz_yr": 50,
        "moly_mlbs_yr": 0,
    },
    "El Campo REE California": {
        "type": "ree",
        "status": "Pre-resource — drilling 2025",
        "resource_mt": 0,
        "grade_cu_pct": 0,
        "contained_cu_kt": 0,
        "contained_au_koz": 0,
        "ownership_pct": 100,
        "target_production_ktpa_cu": 0,
        "first_production_year": 2030,
        "mine_life_years": 20,
        "c1_cash_cost_lb": 0,
        "aisc_lb": 0,
        "capex_total_usdm": 200,
        "capex_sustaining_annual_usdm": 15,
        "royalty_pct": 3.0,
        "tax_rate": 0.27,
        "country": "USA",
        "gold_koz_yr": 0,
        "silver_koz_yr": 0,
        "moly_mlbs_yr": 0,
    },
}

# Project status chart
proj_names  = list(projects.keys())
stages      = ["Inferred MRE", "Pre-Resource Drilling", "Pre-Resource Drilling"]
commodities = ["Cu / Au / Ag", "Sb / Ag", "REE (NdPr)"]
capex_est   = [p['capex_total_usdm'] for p in projects.values()]
risk_score  = [2, 4, 5]  # 1=low, 5=very high
colors_proj = ['#B87333', '#708090', '#4B0082']

fig_proj = go.Figure()
for i, (pname, stage, comm, capex, risk, col) in enumerate(
        zip(proj_names, stages, commodities, capex_est, risk_score, colors_proj)):
    fig_proj.add_trace(go.Bar(
        x=[pname], y=[capex], name=pname,
        marker_color=col,
        text=f"Stage: {stage}<br>Metals: {comm}<br>Est. Capex: USD${capex}M<br>Risk: {'⭐'*risk}",
        textposition='outside'
    ))

fig_proj.update_layout(
    title="<b>Locksley Resources — Project Portfolio</b><br><sub>Estimated development capex; all pre-feasibility stage (ESTIMATED)</sub>",
    yaxis_title="Estimated Development Capex (USD$M)",
    template="plotly_white", showlegend=False, height=450
)
fig_proj.show()


In [7]:
# ── Tottenham Resource Visualisation ────────────────────────────────────────
# In-situ metal value sensitivity chart
cu_prices = np.linspace(3.00, 7.00, 50)
insitu_values = []
for cp in cu_prices:
    cu_val = TOTTENHAM_CU_CONTAINED_KT * 1000 * LBS_PER_TONNE * cp / 1e6
    au_val = TOTTENHAM_AU_CONTAINED_KOZ * GOLD_PRICE_OZ / 1e3
    ag_val = 511 * SILVER_PRICE_OZ / 1e3
    insitu_values.append(cu_val + au_val + ag_val)

fig_insitu = go.Figure()
fig_insitu.add_trace(go.Scatter(
    x=cu_prices, y=insitu_values,
    fill='tozeroy', fillcolor='rgba(184,115,51,0.2)',
    line=dict(color='#B87333', width=3),
    name='Total In-Situ Value (Cu + Au + Ag)'
))
# Add current market cap line
fig_insitu.add_hline(y=MARKET_CAP_USD_M, line_dash='dash', line_color='red',
                      annotation_text=f'Current Mkt Cap USD${MARKET_CAP_USD_M:.0f}M')
# Add current copper price line
idx_live = np.argmin(np.abs(cu_prices - LIVE_COPPER_PRICE_LB))
fig_insitu.add_vline(x=LIVE_COPPER_PRICE_LB, line_dash='dot', line_color='green',
                      annotation_text=f'Live Cu ${LIVE_COPPER_PRICE_LB}/lb')

# By-product breakdown at current price
au_val_curr = TOTTENHAM_AU_CONTAINED_KOZ * GOLD_PRICE_OZ / 1e3
ag_val_curr = 511 * SILVER_PRICE_OZ / 1e3
cu_val_curr = TOTTENHAM_CU_CONTAINED_KT * 1000 * LBS_PER_TONNE * LIVE_COPPER_PRICE_LB / 1e6

print(f"── Tottenham In-Situ Metal Value (@ Live Prices) ──────────")
print(f"Copper  ({TOTTENHAM_CU_CONTAINED_KT:.0f}kt @ ${LIVE_COPPER_PRICE_LB:.2f}/lb):    USD${cu_val_curr:,.1f}M")
print(f"Gold    ({TOTTENHAM_AU_CONTAINED_KOZ:.0f}koz @ ${GOLD_PRICE_OZ}/oz):  USD${au_val_curr:,.1f}M")
print(f"Silver  (511koz @ ${SILVER_PRICE_OZ}/oz):       USD${ag_val_curr:,.1f}M")
print(f"TOTAL IN-SITU VALUE:              USD${cu_val_curr+au_val_curr+ag_val_curr:,.1f}M")
print(f"Market Cap (USD):                 USD${MARKET_CAP_USD_M:.1f}M")
print(f"Mkt Cap / In-Situ Value:         {MARKET_CAP_USD_M/(cu_val_curr+au_val_curr+ag_val_curr)*100:.1f}%")
print(f"(Typical for inferred resource:  2-8% of in-situ)")
print(f"")
print(f"NOTE: Mojave Sb/REE optionality value is NOT included above")

fig_insitu.update_layout(
    title="<b>Tottenham In-Situ Metal Value vs Copper Price</b><br><sub>LKY market cap vs resource value; by-products included at current Au/Ag prices</sub>",
    xaxis_title="Copper Price (USD/lb)", yaxis_title="In-Situ Value (USD$M)",
    template="plotly_white", height=450
)
fig_insitu.show()


── Tottenham In-Situ Metal Value (@ Live Prices) ──────────
Copper  (71kt @ $5.90/lb):    USD$923.4M
Gold    (70koz @ $2900/oz):  USD$202.2M
Silver  (511koz @ $32/oz):       USD$16.4M
TOTAL IN-SITU VALUE:              USD$1,142.0M
Market Cap (USD):                 USD$60.8M
Mkt Cap / In-Situ Value:         5.3%
(Typical for inferred resource:  2-8% of in-situ)

NOTE: Mojave Sb/REE optionality value is NOT included above


## Section 4: Hypothetical Production Model (Tottenham)

> **⚠️ IMPORTANT DISCLAIMER:** All production figures below are **ESTIMATED** based on analogous 
> VMS mines in the Cobar-Girilambone belt (notably Aeris Resources' Tritton Mine). No PEA, PFS 
> or DFS has been completed for Tottenham. These scenarios are **illustrative only** and subject 
> to substantial uncertainty. Actual outcomes depend on resource expansion, metallurgy, permitting, 
> and infrastructure studies that have not yet been conducted.

**Comparable VMS Mines (Girilambone Belt):**
- Aeris Resources' **Tritton Mine** (110km NW of Tottenham): ~23ktpa Cu, C1 ~$1.40/lb, A$300M project
- Helix Resources' **CZ Copper Deposit** (adjacent to Tottenham): pre-development peer
- Constellation Resources (adjacent corridor): newly discovered, similar geology

**Assumed Scenario Parameters for Tottenham:**

| Parameter | Conservative | Base | Optimistic |
|-----------|-------------|------|------------|
| Mine scale | 8 ktpa Cu | 12 ktpa Cu | 20 ktpa Cu |
| Resource (Mt) | 9.86 (current) | 15 (expansion) | 40 (50km corridor) |
| C1 Cash Cost | $2.20/lb | $1.80/lb | $1.40/lb |
| AISC | $3.00/lb | $2.50/lb | $2.00/lb |
| First Production | 2033 | 2031 | 2030 |
| Dev Capex | $180M | $150M | $120M |
| Mine Life | 12 years | 15 years | 25+ years |


In [8]:
# ── Hypothetical Tottenham Production Schedule ────────────────────────────
proj_years = list(range(2025, 2047))
n_years = len(proj_years)

scenarios = {
    'Bear (Conservative)': {
        'start': 2033, 'ramp': 2, 'capacity_ktpa': 8,
        'c1_lb': 2.20, 'aisc_lb': 3.00, 'capex_m': 180, 'life': 12,
        'au_koz_yr': 5, 'ag_koz_yr': 50, 'color': '#F44336'
    },
    'Base Case': {
        'start': 2031, 'ramp': 2, 'capacity_ktpa': 12,
        'c1_lb': 1.80, 'aisc_lb': 2.50, 'capex_m': 150, 'life': 15,
        'au_koz_yr': 8, 'ag_koz_yr': 75, 'color': '#B87333'
    },
    'Bull (Optimistic)': {
        'start': 2030, 'ramp': 2, 'capacity_ktpa': 20,
        'c1_lb': 1.40, 'aisc_lb': 2.00, 'capex_m': 120, 'life': 22,
        'au_koz_yr': 15, 'ag_koz_yr': 125, 'color': '#4CAF50'
    },
}

fig_prod = go.Figure()
for sc_name, sc in scenarios.items():
    prod = []
    for yr in proj_years:
        if yr < sc['start']:
            prod.append(0)
        elif yr < sc['start'] + sc['ramp']:
            prod.append(sc['capacity_ktpa'] * (yr - sc['start'] + 1) / sc['ramp'])
        elif yr < sc['start'] + sc['life']:
            prod.append(sc['capacity_ktpa'])
        else:
            prod.append(0)
    fig_prod.add_trace(go.Scatter(
        x=proj_years, y=prod, name=sc_name,
        line=dict(color=sc['color'], width=2.5),
        fill='tozeroy' if sc_name == 'Base Case' else None,
        fillcolor='rgba(184,115,51,0.15)' if sc_name == 'Base Case' else None
    ))

fig_prod.add_vline(x=2026, line_dash='dot', line_color='gray',
                   annotation_text='Today (2026)')
fig_prod.update_layout(
    title="<b>Tottenham — Hypothetical Cu Production Schedule (ESTIMATED)</b><br><sub>Pre-PEA estimates only; actual production depends on resource expansion, permitting, financing</sub>",
    xaxis_title="Year", yaxis_title="Cu Production (ktpa)",
    template="plotly_white", hovermode="x unified", height=450
)
fig_prod.show()

print("ESTIMATED: All production scenarios — indicative only, no PEA/PFS completed")
print(f"Base case peak: {scenarios['Base Case']['capacity_ktpa']} ktpa Cu from ~{scenarios['Base Case']['start']}")
print(f"That's {scenarios['Base Case']['capacity_ktpa']/1000*1e6:.0f} tonnes per annum of refined copper equivalent")


ESTIMATED: All production scenarios — indicative only, no PEA/PFS completed
Base case peak: 12 ktpa Cu from ~2031
That's 12000 tonnes per annum of refined copper equivalent


## Section 5: Mine-Level Economics (Base Case — ESTIMATED)

In [9]:
# ── Tottenham Mine P&L Model (Base Case, ESTIMATED) ─────────────────────────
sc = scenarios['Base Case']
mine_years = list(range(sc['start'], sc['start'] + sc['life']))
mine_prod_ktpa = [0]*2 + [sc['capacity_ktpa']] * (sc['life'] - 2)

results_base = []
for i, yr in enumerate(mine_years):
    prod_frac = min(1.0, (i+1) / sc['ramp'])
    cu_prod_kt = sc['capacity_ktpa'] * prod_frac
    
    for cu_price, label in [(COPPER_PRICE_BEAR_LB,'Bear'), 
                             (COPPER_PRICE_BASE_LB,'Base'), 
                             (COPPER_PRICE_BULL_LB,'Bull')]:
        cu_rev = cu_prod_kt * 1000 * LBS_PER_TONNE * cu_price / 1e6
        au_rev = sc['au_koz_yr'] * prod_frac * GOLD_PRICE_OZ / 1e3
        ag_rev = sc['ag_koz_yr'] * prod_frac * SILVER_PRICE_OZ / 1e3
        total_rev = cu_rev + au_rev + ag_rev
        
        # AISC costs
        aisc_total = cu_prod_kt * 1000 * LBS_PER_TONNE * sc['aisc_lb'] / 1e6
        royalties = total_rev * 0.025  # 2.5% NSW royalty
        sustaining = sc['capex_m'] * 0.05 * prod_frac  # 5% of dev capex/yr sustaining
        
        ebitda = total_rev - aisc_total - royalties
        ebit = ebitda - sustaining
        tax = max(0, ebit * 0.30)
        fcf = ebit - tax
        
        results_base.append({
            'year': yr, 'scenario': label,
            'cu_prod_kt': round(cu_prod_kt, 2),
            'cu_revenue_usdm': round(cu_rev, 2),
            'au_ag_revenue_usdm': round(au_rev + ag_rev, 2),
            'total_revenue_usdm': round(total_rev, 2),
            'aisc_costs_usdm': round(aisc_total, 2),
            'ebitda_usdm': round(ebitda, 2),
            'fcf_usdm': round(fcf, 2),
            'ebitda_margin_pct': round(ebitda/total_rev*100, 1) if total_rev > 0 else 0
        })

df_mine = pd.DataFrame(results_base)
df_base = df_mine[df_mine['scenario'] == 'Base'].copy()
df_bear = df_mine[df_mine['scenario'] == 'Bear'].copy()
df_bull = df_mine[df_mine['scenario'] == 'Bull'].copy()

fig_ebitda = go.Figure()
fig_ebitda.add_trace(go.Scatter(x=df_bull['year'], y=df_bull['ebitda_usdm'], 
                                 fill='tonexty', name='Bull EBITDA', 
                                 line=dict(color='#4CAF50', dash='dot')))
fig_ebitda.add_trace(go.Scatter(x=df_base['year'], y=df_base['ebitda_usdm'],
                                 fill='tozeroy', fillcolor='rgba(184,115,51,0.3)',
                                 name='Base EBITDA', line=dict(color='#B87333', width=3)))
fig_ebitda.add_trace(go.Scatter(x=df_bear['year'], y=df_bear['ebitda_usdm'],
                                 name='Bear EBITDA', line=dict(color='#F44336', dash='dot')))

fig_ebitda.update_layout(
    title="<b>Tottenham EBITDA by Scenario — Hypothetical Mine (ESTIMATED)</b><br><sub>Base: ${:.2f}/lb Cu | Bear: ${:.2f}/lb | Bull: ${:.2f}/lb</sub>".format(
        COPPER_PRICE_BASE_LB, COPPER_PRICE_BEAR_LB, COPPER_PRICE_BULL_LB),
    xaxis_title="Year", yaxis_title="EBITDA (USD$M)",
    template="plotly_white", height=450
)
fig_ebitda.show()

print(f"Base Case Peak EBITDA:  USD${df_base['ebitda_usdm'].max():.1f}M/yr")
print(f"Bear Case Peak EBITDA:  USD${df_bear['ebitda_usdm'].max():.1f}M/yr")
print(f"Bull Case Peak EBITDA:  USD${df_bull['ebitda_usdm'].max():.1f}M/yr")
print(f"Base EBITDA Margin avg: {df_base['ebitda_margin_pct'].mean():.1f}%")
print("⚠️ ESTIMATED — pre-PEA; costs subject to material revision")


Base Case Peak EBITDA:  USD$87.8M/yr
Bear Case Peak EBITDA:  USD$55.5M/yr
Bull Case Peak EBITDA:  USD$126.5M/yr
Base EBITDA Margin avg: 55.6%
⚠️ ESTIMATED — pre-PEA; costs subject to material revision


## Section 6: P&L Waterfall — Base Case Year 3 (ESTIMATED)

In [10]:
# ── P&L Waterfall (Base Case, Year 3 of Production) ──────────────────────────
try:
    row = df_base.iloc[2]  # Year 3 of production (full nameplate)
    cu_rev    = row['cu_revenue_usdm']
    bp_rev    = row['au_ag_revenue_usdm']
    total_rev = row['total_revenue_usdm']
    costs     = -row['aisc_costs_usdm']
    royalties = -total_rev * 0.025
    ebitda    = row['ebitda_usdm']
    sustain   = -sc['capex_m'] * 0.05
    ebit      = ebitda + sustain
    tax       = -max(0, ebit * 0.30)
    fcf       = ebit + tax

    labels = ['Cu Revenue','By-Products','Total Revenue','AISC Costs','Royalties',
              'EBITDA','Sustaining Capex','EBIT','Tax','FCF']
    values = [cu_rev, bp_rev, total_rev, costs, royalties, ebitda, sustain, ebit, tax, fcf]
    measures=['absolute','relative','total','relative','relative','total',
              'relative','total','relative','total']

    fig_wf = go.Figure(go.Waterfall(
        orientation="v", measure=measures, x=labels, y=values,
        connector={"line": {"color": "rgb(63, 63, 63)"}},
        increasing={"marker": {"color": "#4CAF50"}},
        decreasing={"marker": {"color": "#F44336"}},
        totals={"marker": {"color": "#B87333"}}
    ))
    fig_wf.update_layout(
        title=f"<b>Tottenham P&L Bridge — Year 3 Production, Base Case (ESTIMATED)</b><br><sub>Cu @ ${COPPER_PRICE_BASE_LB}/lb | Au @ ${GOLD_PRICE_OZ}/oz | {sc['capacity_ktpa']}ktpa output</sub>",
        yaxis_title="USD$M", template="plotly_white", height=500
    )
    fig_wf.show()
    
    print(f"Year 3 (Full Nameplate) — Base Case:")
    print(f"  Revenue:    USD${total_rev:.1f}M")
    print(f"  EBITDA:     USD${ebitda:.1f}M  ({row['ebitda_margin_pct']:.1f}% margin)")
    print(f"  FCF:        USD${fcf:.1f}M")
    print(f"  Cu revenue: {cu_rev/total_rev*100:.1f}% | By-products: {bp_rev/total_rev*100:.1f}%")
except Exception as e:
    print(f"Waterfall error: {e}")


Year 3 (Full Nameplate) — Base Case:
  Revenue:    USD$157.9M
  EBITDA:     USD$87.8M  (55.6% margin)
  FCF:        USD$56.2M
  Cu revenue: 83.8% | By-products: 16.2%


## Section 7: DCF Valuation — Exploration Risk-Adjusted NAV

### Methodology

For exploration-stage companies, a standard DCF is applied with **two risk-adjustment layers**:

1. **Unadjusted NPV**: Present value of future mine cash flows, discounted at WACC
2. **Risk-Adjusted NAV (rNAV)**: NPV × probability of success
   - P(exploration success) = 30% (resource → economic deposit)
   - P(development) = 50% (economic deposit → producing mine)
   - **Combined P(success) = 15%** — this is conservative for a JORC resource holder

3. **Option Value for Mojave**: Separate binary probability scenario for antimony/REE

> **⚠️ ALL DCF VALUES ARE ESTIMATED** based on assumed mine parameters. 
> Tottenham has no PEA/PFS/DFS. Treat these as indicative ranges only.


In [11]:
# ── DCF Valuation Function ──────────────────────────────────────────────────
def dcf_valuation(copper_price_lb, scenario_label="Base", 
                  mine_capacity_kt=12, c1_lb=1.80, aisc_lb=2.50,
                  start_yr=2031, mine_life=15, capex_m=150,
                  wacc=WACC, au_koz=8, ag_koz=75):
    # Risk-adjusted DCF for Tottenham prospect.
    years_range = list(range(start_yr, start_yr + mine_life))
    fcfs = []
    
    for i, yr in enumerate(years_range):
        prod_frac = min(1.0, (i+1) / 2)
        cu_kt = mine_capacity_kt * prod_frac
        
        cu_rev  = cu_kt * 1000 * LBS_PER_TONNE * copper_price_lb / 1e6
        au_rev  = au_koz * prod_frac * GOLD_PRICE_OZ / 1e3
        ag_rev  = ag_koz * prod_frac * SILVER_PRICE_OZ / 1e3
        total_r = cu_rev + au_rev + ag_rev
        
        aisc_c  = cu_kt * 1000 * LBS_PER_TONNE * aisc_lb / 1e6
        royalty = total_r * 0.025
        sustain = capex_m * 0.05 * prod_frac
        ebitda  = total_r - aisc_c - royalty
        ebit    = ebitda - sustain
        tax     = max(0, ebit * 0.30)
        fcf     = ebit - tax - GA_BASE_USD_M * 0.3  # share of G&A
        fcfs.append(fcf)
    
    # Pre-production capex (development period)
    yrs_from_now = start_yr - 2026
    
    # Discount FCFs
    discount_factors = []
    for i in range(mine_life):
        yr_from_now = yrs_from_now + i
        discount_factors.append(1 / (1 + wacc) ** yr_from_now)
    
    pv_fcfs = [f * d for f, d in zip(fcfs, discount_factors)]
    
    # Terminal value
    tv = fcfs[-1] * (1 + TERMINAL_GROWTH) / (wacc - TERMINAL_GROWTH)
    pv_tv = tv * discount_factors[-1]
    
    # Dev capex PV (negative)
    pv_capex = -capex_m / (1 + wacc) ** (yrs_from_now - 2)  # mid-way through dev
    
    enterprise_value = sum(pv_fcfs) + pv_tv + pv_capex
    equity_value     = enterprise_value - NET_DEBT_USD_M
    nav_per_share    = equity_value / SHARES_OUT_M
    
    # Risk-adjusted values
    rnav_ev          = enterprise_value * COMBINED_SUCCESS
    rnav_equity      = equity_value * COMBINED_SUCCESS
    rnav_per_share   = nav_per_share * COMBINED_SUCCESS
    p_rnav           = (CURRENT_PRICE_USD / rnav_per_share) if rnav_per_share > 0 else 0
    
    return {
        'scenario': scenario_label,
        'copper_lb': copper_price_lb,
        'ev_usdm': round(enterprise_value, 1),
        'equity_usdm': round(equity_value, 1),
        'nav_share': round(nav_per_share, 3),
        'rnav_ev_usdm': round(rnav_ev, 1),
        'rnav_equity_usdm': round(rnav_equity, 1),
        'rnav_share': round(rnav_per_share, 3),
        'p_rnav': round(p_rnav, 2),
        'pv_fcfs_usdm': round(sum(pv_fcfs), 1),
        'pv_tv_usdm': round(pv_tv, 1),
        'pv_capex_usdm': round(pv_capex, 1),
    }

# Run all three scenarios
results = {
    'bear': dcf_valuation(COPPER_PRICE_BEAR_LB, "Bear (Conservative)",
                          mine_capacity_kt=8, c1_lb=2.20, aisc_lb=3.00,
                          start_yr=2033, mine_life=12, capex_m=180),
    'base': dcf_valuation(COPPER_PRICE_BASE_LB, "Base Case",
                          mine_capacity_kt=12, c1_lb=1.80, aisc_lb=2.50,
                          start_yr=2031, mine_life=15, capex_m=150),
    'bull': dcf_valuation(COPPER_PRICE_BULL_LB, "Bull (Optimistic)",
                          mine_capacity_kt=20, c1_lb=1.40, aisc_lb=2.00,
                          start_yr=2030, mine_life=22, capex_m=120),
}

print("=" * 80)
print("  TOTTENHAM PROJECT — DCF VALUATION (ESTIMATED, EXPLORATION RISK-ADJUSTED)")
print("  ⚠️  No PEA/PFS/DFS completed. All figures are indicative scenarios only.")
print("=" * 80)
print(f"{'Scenario':<25} {'Cu $/lb':<10} {'EV ($M)':<12} {'Equity ($M)':<14} {'NAV/sh':<10} {'rNAV/sh':<12} {'P/rNAV'}")
print("-" * 80)
for k, r in results.items():
    print(f"{r['scenario']:<25} ${r['copper_lb']:<9.2f} ${r['ev_usdm']:<11.0f} ${r['equity_usdm']:<13.0f} ${r['nav_share']:<9.3f} ${r['rnav_share']:<11.3f} {r['p_rnav']:.2f}x")
print("-" * 80)
print(f"Current price (USD): ${CURRENT_PRICE_USD:.3f}/share")
print(f"Risk-adj probability: {COMBINED_SUCCESS*100:.0f}% ({EXPLORATION_RISK*100:.0f}% exploration × {DEVELOPMENT_RISK*100:.0f}% development)")
print()
print("KEY INSIGHT: P/rNAV > 1.0x means market is pricing in:")
print("  (a) higher probability of success than 15%, or")
print("  (b) Mojave antimony/REE optionality premium above Tottenham Cu value, or")
print("  (c) speculative re-rating premium")


  TOTTENHAM PROJECT — DCF VALUATION (ESTIMATED, EXPLORATION RISK-ADJUSTED)
  ⚠️  No PEA/PFS/DFS completed. All figures are indicative scenarios only.
Scenario                  Cu $/lb    EV ($M)      Equity ($M)    NAV/sh     rNAV/sh      P/rNAV
--------------------------------------------------------------------------------
Bear (Conservative)       $3.75      $-26         $-22           $-0.076    $-0.011      0.00x
Base Case                 $5.00      $180         $184           $0.640     $0.096       2.20x
Bull (Optimistic)         $6.50      $739         $743           $2.580     $0.387       0.55x
--------------------------------------------------------------------------------
Current price (USD): $0.211/share
Risk-adj probability: 15% (30% exploration × 50% development)

KEY INSIGHT: P/rNAV > 1.0x means market is pricing in:
  (a) higher probability of success than 15%, or
  (b) Mojave antimony/REE optionality premium above Tottenham Cu value, or
  (c) speculative re-rating pre

In [12]:
# ── Sensitivity: NPV vs WACC × Copper Price ──────────────────────────────────
wacc_range  = [0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15]
price_range = [3.00, 3.50, 4.00, 4.50, 5.00, 5.50, 6.00, 6.61]

matrix = []
for w in wacc_range:
    row_vals = []
    for p in price_range:
        r = dcf_valuation(p, wacc=w, mine_capacity_kt=12, c1_lb=1.80, aisc_lb=2.50,
                          start_yr=2031, mine_life=15, capex_m=150)
        row_vals.append(round(r['ev_usdm'], 0))
    matrix.append(row_vals)

df_sens = pd.DataFrame(matrix,
    index=[f"{int(w*100)}%" for w in wacc_range],
    columns=[f"${p:.2f}" for p in price_range])

fig_sens = px.imshow(df_sens, text_auto=True, color_continuous_scale='RdYlGn',
    title="<b>Tottenham Unadjusted EV (USD$M) — WACC vs Copper Price Sensitivity</b><br><sub>Base case mine params | Multiply by 15% for risk-adjusted value (ESTIMATED)</sub>",
    labels=dict(x="Copper Price (USD/lb)", y="WACC"))
fig_sens.update_layout(height=450)
fig_sens.show()

# Risk-adjusted version
matrix_r = [[v * COMBINED_SUCCESS for v in row] for row in matrix]
df_rsens = pd.DataFrame(matrix_r,
    index=[f"{int(w*100)}%" for w in wacc_range],
    columns=[f"${p:.2f}" for p in price_range])

fig_rsens = px.imshow(df_rsens, text_auto=True, color_continuous_scale='RdYlGn',
    title="<b>Tottenham Risk-Adjusted EV (USD$M) — 15% P(success) × Unadjusted NPV</b><br><sub>Represents expected value; market cap to compare = USD${:.0f}M</sub>".format(MARKET_CAP_USD_M),
    labels=dict(x="Copper Price (USD/lb)", y="WACC"))
fig_rsens.update_layout(height=450)
fig_rsens.show()


## Section 8: Full Valuation Summary — Tottenham + Mojave Optionality

In [13]:
# ── Full Valuation Waterfall incl. Mojave Optionality ─────────────────────────
# Sum of parts: Tottenham rNAV + Mojave antimony option + REE option + cash

# Mojave Antimony Value (binary option)
# If drilling confirms resource → potential value (comparable: Perpetua Resources, US Antimony)
# US antimony market: ~30M lbs/yr demand, zero domestic supply, ~$25/lb
# Conservative: small mine 2-3M lbs/yr → revenue ~$50-75M/yr, 30% EBITDA → $15-22M/yr FCF
# Fully-valued: USD$50-150M; risk-adjusted 20% P(success) → USD$10-30M
MOJAVE_SB_OPTION_USDM_BASE   = 20.0   # USD$M risk-adjusted (ESTIMATED)
MOJAVE_SB_OPTION_USDM_BULL   = 60.0   # If drilling hits resource & DoD interest confirmed
MOJAVE_REE_OPTION_USDM_BASE  = 10.0   # Very early stage; highly speculative
MOJAVE_REE_OPTION_USDM_BULL  = 40.0   # Adjacent to Mountain Pass; strategic premium
CASH_USD_M                   = 6.0 * AUD_USD  # ~USD$3.8M

sum_of_parts = {
    'Component': [
        'Tottenham rNAV Bear',
        'Tottenham rNAV Base',
        'Tottenham rNAV Bull',
        'Mojave Sb Base',
        'Mojave Sb Bull',
        'Mojave REE Base',
        'Mojave REE Bull',
        'Net Cash',
    ],
    'Value_USD_M': [
        results['bear']['rnav_ev_usdm'],
        results['base']['rnav_ev_usdm'],
        results['bull']['rnav_ev_usdm'],
        MOJAVE_SB_OPTION_USDM_BASE,
        MOJAVE_SB_OPTION_USDM_BULL,
        MOJAVE_REE_OPTION_USDM_BASE,
        MOJAVE_REE_OPTION_USDM_BULL,
        CASH_USD_M,
    ]
}

# Scenario combinations
scenarios_sotp = {
    'Bear All': results['bear']['rnav_ev_usdm'] + MOJAVE_SB_OPTION_USDM_BASE + MOJAVE_REE_OPTION_USDM_BASE + CASH_USD_M,
    'Base All': results['base']['rnav_ev_usdm'] + MOJAVE_SB_OPTION_USDM_BASE + MOJAVE_REE_OPTION_USDM_BASE + CASH_USD_M,
    'Bull All': results['bull']['rnav_ev_usdm'] + MOJAVE_SB_OPTION_USDM_BULL + MOJAVE_REE_OPTION_USDM_BULL + CASH_USD_M,
    'Current Mkt Cap': MARKET_CAP_USD_M,
}

fig_sotp = go.Figure(go.Bar(
    x=list(scenarios_sotp.keys()),
    y=list(scenarios_sotp.values()),
    marker_color=['#F44336','#FF9800','#4CAF50','#1565C0'],
    text=[f"USD${v:.0f}M | {v/MARKET_CAP_USD_M*100:.0f}% of Cap" for v in scenarios_sotp.values()],
    textposition='outside'
))
fig_sotp.add_hline(y=MARKET_CAP_USD_M, line_dash='dash', line_color='blue',
                    annotation_text=f'Current Market Cap USD${MARKET_CAP_USD_M:.0f}M')
fig_sotp.update_layout(
    title="<b>LKY Sum-of-Parts Valuation — Bear / Base / Bull (ESTIMATED)</b><br><sub>Risk-adjusted rNAV for Tottenham + option value for Mojave assets</sub>",
    yaxis_title="USD$M", template="plotly_white", height=500
)
fig_sotp.show()

print("=" * 65)
print("  SUM-OF-PARTS VALUATION SUMMARY (ESTIMATED)")
print("=" * 65)
for name, val in scenarios_sotp.items():
    implied_share = val / SHARES_OUT_M * (1/AUD_USD)
    marker = " ← CURRENT" if name == 'Current Mkt Cap' else ""
    print(f"  {name:<20}: USD${val:>6.0f}M = A${implied_share:.3f}/share{marker}")
print()
print(f"  Current price: A${CURRENT_PRICE_AUD:.3f}")
print(f"  Premium to Bear rNAV:  {MARKET_CAP_USD_M / scenarios_sotp['Bear All']:.1f}x")
print(f"  Premium to Base rNAV:  {MARKET_CAP_USD_M / scenarios_sotp['Base All']:.1f}x")
print(f"  Premium to Bull rNAV:  {MARKET_CAP_USD_M / scenarios_sotp['Bull All']:.1f}x")
print()
print("  NOTE: Premium to rNAV is EXPECTED for early-stage junior")
print("  explorers due to exploration optionality and momentum.")
print("  A P/rNAV of 1-4x is typical for quality juniors with catalysts.")
print("=" * 65)


  SUM-OF-PARTS VALUATION SUMMARY (ESTIMATED)
  Bear All            : USD$    30M = A$0.165/share
  Base All            : USD$    61M = A$0.336/share
  Bull All            : USD$   215M = A$1.183/share
  Current Mkt Cap     : USD$    61M = A$0.335/share ← CURRENT

  Current price: A$0.335
  Premium to Bear rNAV:  2.0x
  Premium to Base rNAV:  1.0x
  Premium to Bull rNAV:  0.3x

  NOTE: Premium to rNAV is EXPECTED for early-stage junior
  explorers due to exploration optionality and momentum.
  A P/rNAV of 1-4x is typical for quality juniors with catalysts.


## Section 9: Peer Comparison

### Comparable Companies & Market Context

Locksley is compared across two peer groups:

1. **Copper Explorer Peers (Girilambone Belt, ASX)** — resource-stage copper juniors in NSW
2. **US Critical Minerals Plays (ASX/TSX/NYSE)** — antimony/REE explorers with US government alignment


In [14]:
# ── Peer Comparison Table ──────────────────────────────────────────────────
peers_data = {
    'Company': [
        'Locksley Resources (LKY)', 
        'Helix Resources (HLX)',
        'Bacchus Resources',
        'Aeris Resources (AIS) — Producer',
        'Perpetua Resources (PPTA) — Sb/Au',
        'NovaBay (Antimony Pure Play)',
        'MP Materials (MP) — REE Producer'
    ],
    'Ticker': ['LKY.ASX','HLX.ASX','Private','AIS.ASX','PPTA.NASDAQ','N/A','MP.NYSE'],
    'Stage': ['Inferred MRE', 'Exploration', 'Exploration', 'Producer', 'PFS/Permitted', 'Exploration', 'Producer'],
    'Key Metal': ['Cu/Au/Sb/REE','Cu','Cu/Au','Cu','Sb/Au','Sb','REE'],
    'Location': ['NSW+California','NSW','NSW','NSW','Idaho, USA','USA','California, USA'],
    'Market Cap (AUD M)': [97, 18, 'N/A', 45, 580, 'N/A', 4800],
    'Cu Resource (kt)': [71, 25, 10, 195, 0, 0, 0],
    'Notes': [
        'Dual Cu/Sb-REE; US critical minerals focus',
        'CZ Deposit adjacent Tottenham; single asset',
        'Private; historic exploration only',
        'Tritton Mine — 23ktpa Cu; benchmark producer',
        'Only Sb project in US with DoD backing; Idaho',
        'Pre-resource antimony — speculative',
        'Only US REE producer; benchmark'
    ]
}
df_peers = pd.DataFrame(peers_data)

# In-situ value comparison (Cu plays only)
cu_peers = {
    'Locksley (LKY)': {'mktcap_usd': MARKET_CAP_USD_M, 'cu_kt': TOTTENHAM_CU_CONTAINED_KT, 'stage': 'Inferred'},
    'Helix (HLX)':    {'mktcap_usd': 18*AUD_USD, 'cu_kt': 25, 'stage': 'Exploration'},
    'Aeris (AIS)':    {'mktcap_usd': 45*AUD_USD, 'cu_kt': 195, 'stage': 'Producer'},
}

ev_per_kt = {k: v['mktcap_usd']/v['cu_kt'] for k, v in cu_peers.items()}
colors_peer = ['#B87333','#9E9E9E','#4CAF50']

fig_peer = go.Figure()
fig_peer.add_trace(go.Bar(
    x=list(ev_per_kt.keys()), y=list(ev_per_kt.values()),
    marker_color=colors_peer,
    text=[f"USD${v:,.0f}/t" for v in ev_per_kt.values()],
    textposition='outside'
))
fig_peer.update_layout(
    title="<b>Market Cap per Tonne of Cu In-Situ — Peer Comparison</b><br><sub>Higher = more expensive relative to resource; explorer premium vs producer typical</sub>",
    yaxis_title="USD$/tonne Cu in-situ",
    template="plotly_white", height=420
)
fig_peer.show()

print("Peer Comparison Table:")
print(df_peers[['Company','Stage','Key Metal','Market Cap (AUD M)','Cu Resource (kt)']].to_string(index=False))
print()
print("EV/Resource (USD$/tonne Cu in-situ):")
for comp, val in ev_per_kt.items():
    print(f"  {comp:<20}: USD${val:,.0f}/t")
print()
print("Typical ranges: Producing mines $100-400/t | Advanced explorer $50-200/t | Early explorer $20-100/t")
print(f"LKY (Cu only basis): USD${ev_per_kt['Locksley (LKY)']:,.0f}/t — elevated due to Mojave premium")


Peer Comparison Table:
                          Company         Stage    Key Metal Market Cap (AUD M)  Cu Resource (kt)
         Locksley Resources (LKY)  Inferred MRE Cu/Au/Sb/REE                 97                71
            Helix Resources (HLX)   Exploration           Cu                 18                25
                Bacchus Resources   Exploration        Cu/Au                N/A                10
 Aeris Resources (AIS) — Producer      Producer           Cu                 45               195
Perpetua Resources (PPTA) — Sb/Au PFS/Permitted        Sb/Au                580                 0
     NovaBay (Antimony Pure Play)   Exploration           Sb                N/A                 0
 MP Materials (MP) — REE Producer      Producer          REE               4800                 0

EV/Resource (USD$/tonne Cu in-situ):
  Locksley (LKY)      : USD$1/t
  Helix (HLX)         : USD$0/t
  Aeris (AIS)         : USD$0/t

Typical ranges: Producing mines $100-400/t | Advanced expl

In [15]:
# ── US Critical Minerals Valuation Benchmark ──────────────────────────────
print("╔══════════════════════════════════════════════════════════════════╗")
print("║  US CRITICAL MINERALS CONTEXT — ANTIMONY MARKET                 ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  Global antimony supply:  ~170,000 tpa (2024)                   ║")
print("║  China's share:           >90% (export controls since Aug 2024)  ║")
print("║  US antimony production:  ZERO — 100% import dependent          ║")
print("║  US annual demand:        ~30M lbs (defence, flame retardants,  ║")
print("║                           batteries, semiconductors)             ║")
print("║  Current antimony price:  ~$25/lb (2-3x pre-restriction)        ║")
print("║  LKY DAM surface grade:   UP TO 46% Sb (world-class grade)     ║")
print("║  Comparable: Perpetua Resources — DoD backed $1.8B Sb project  ║")
print("║               (Idaho); market cap ~$580M AUD vs LKY $97M AUD   ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  KEY RISK: Antimony resource NOT yet defined (drilling 2025)    ║")
print("║  The market is pricing in potential, not proven resource        ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()
print("US REE Market Context:")
print(f"  Mountain Pass (MP Materials): Only US REE producer; ~US$500M Apple deal, US$400M DoD")
print(f"  El Campo: 1.4km from Mountain Pass; 12.1% TREO, 3.19% NdPr")
print(f"  NdPr price: ~$65/kg — critical for EV magnets, wind turbines, defence")
print(f"  China REE export controls: Dy/Tb prices 2-3x pre-restriction levels")
print(f"  LKY is EARLIEST STAGE (no JORC resource yet) — maximum optionality, maximum risk")


╔══════════════════════════════════════════════════════════════════╗
║  US CRITICAL MINERALS CONTEXT — ANTIMONY MARKET                 ║
╠══════════════════════════════════════════════════════════════════╣
║  Global antimony supply:  ~170,000 tpa (2024)                   ║
║  China's share:           >90% (export controls since Aug 2024)  ║
║  US antimony production:  ZERO — 100% import dependent          ║
║  US annual demand:        ~30M lbs (defence, flame retardants,  ║
║                           batteries, semiconductors)             ║
║  Current antimony price:  ~$25/lb (2-3x pre-restriction)        ║
║  LKY DAM surface grade:   UP TO 46% Sb (world-class grade)     ║
║  Comparable: Perpetua Resources — DoD backed $1.8B Sb project  ║
║               (Idaho); market cap ~$580M AUD vs LKY $97M AUD   ║
╠══════════════════════════════════════════════════════════════════╣
║  KEY RISK: Antimony resource NOT yet defined (drilling 2025)    ║
║  The market is pricing in potential, not pro

## Section 10: Investment Thesis, Risks & Key Catalysts

---

## 🟢 BULL CASE — Why LKY Could Be Worth Significantly More

### 1. Copper Supercycle at the Most Important Inflection Point in History
- Copper hit **$6.61/lb on January 29, 2026** — a new all-time record
- Structural deficits forecast from 2027+ as EV/grid/AI demand outpaces mine supply
- Tottenham's 50km VHMS corridor is genuinely under-explored — the maiden MRE of 9.86Mt barely scratches the surface
- **Grade of 0.72% Cu** is above typical open-pit threshold; high-grade veins (5-10% Cu) could support underground mining
- **Cobar-Girilambone belt has produced multiple commercial deposits** — Tritton (Aeris), Girilambone — Tottenham has always been the "sleeping giant"

### 2. US Antimony — A Perfect Policy Storm
- **China controls >90% of global antimony supply** and implemented export controls in August 2024
- The US has **zero domestic antimony production** — a vulnerability that DoD classifies as critical
- **Perpetua Resources** (Idaho Au-Sb) received DoD backing at a >$1.8B project value; LKY at $97M AUD is 20x cheaper
- LKY's Desert Antimony Mine has **surface grades of 46% Sb** — among the highest globally
- The Rice University R&D agreement + GreenMet + EXIM Bank LOI constitute a genuine "government-pathway" strategy

### 3. REE Strategic Premium
- **1.4km from Mountain Pass Mine** — the only producing REE mine in North America
- NdPr grades of 3.19% (high-value rare earths) at El Campo
- Apple ($500M) and DoD ($400M) invested in MP Materials — demonstrates strategic premium
- Any JORC resource confirmation could trigger substantial re-rating

### 4. Management Uplift
- New CEO (Kerrie Matthews) with EPCM/mining execution background (WSP, Fortescue, Mineral Resources)
- New COO (Danny George) with major project delivery experience
- Strategic advisor: Tribeca Capital (deep critical minerals expertise)

---

## 🔴 BEAR CASE — Key Risks

### 1. Pre-Resource Stage — All Value is Speculative
- **No NI 43-101 / JORC resource for Mojave Sb or REE** — the most critical upcoming catalysts could disappoint
- Desert Antimony Mine: historical grades ≠ JORC resource; surface samples are cherry-picked best cases
- **If maiden Mojave drilling fails to deliver high-grade results at depth**, the narrative collapses

### 2. Capital Burn & Dilution Risk
- Cash burn ~A$6M/yr vs ~A$6M cash = **funded for <12 months at current rate**
- Exploration companies must continually raise capital → **systematic dilution** for existing shareholders
- Simply Wall St notes: "*shareholders have been substantially diluted in the past year*"
- To get to production at Tottenham requires A$150-200M+ capex — current market cap insufficient

### 3. Copper Price Reversal
- At $5.90/lb, copper is pricing in near-perfect forward demand
- **Goldman Sachs estimates 600,000t copper surplus in 2025** — prices running ahead of fundamentals
- If China slowdown deepens, copper could correct sharply (precedent: 2022 crash from $4.80 → $3.15)

### 4. US Permitting & BLM Risk
- Mojave is federal land managed by Bureau of Land Management
- Environmental opposition, water rights, and permitting timelines are unpredictable
- California's regulatory environment is among the most complex in the world

### 5. Tottenham Project Risks
- Resource expansion not guaranteed — off-hole DHEM conductors may not drill out
- NSW tenements expire 2026 — renewal risk (standard but administrative burden)
- Distance from infrastructure: processing facilities, power, water in remote central NSW

### 6. Jurisdictional & FX Risk
- Split jurisdiction: AUD projects (Tottenham) + USD projects (Mojave) with A$/USD FX exposure
- Exploration costs in AUD but commodity revenues would be in USD — natural hedge is partial

---

## 🎯 KEY CATALYSTS TO WATCH

| Catalyst | Timeline | Impact |
|----------|----------|--------|
| Mojave DAM maiden drill results | Q4 2025 – Q1 2026 | HIGH — binary event |
| El Campo REE drill results | Q1 2026 | HIGH — binary event |
| Tottenham updated resource estimate | Q4 2025 – Q1 2026 | MEDIUM-HIGH |
| DoD/EXIM Bank formal grant/loan | 2026 | HIGH — non-dilutive capital |
| Rice University processing breakthrough | 2026-2027 | MEDIUM |
| Tottenham PEA/PFS initiation | 2027+ | MEDIUM (longer-term) |
| Copper price movement (Trump tariffs, China demand) | Continuous | HIGH |

---

## 📊 SUMMARY INVESTMENT SCORECARD

| Criterion | Score | Comment |
|-----------|-------|---------|
| Commodity Exposure | ⭐⭐⭐⭐⭐ | Copper + Antimony + REE — triple exposure to critical minerals supercycle |
| Resource Quality | ⭐⭐⭐ | JORC inferred Cu-Au (solid); Mojave pre-resource (speculative) |
| Management Quality | ⭐⭐⭐⭐ | Upgraded in 2025; strong operational + strategic capability |
| Strategic Positioning | ⭐⭐⭐⭐⭐ | Adjacent MP Materials; DoD/DoE alignment; Rice University |
| Funding/Dilution Risk | ⭐⭐ | Cash burn concern; high dilution history |
| Development Risk | ⭐⭐⭐ | Multiple pre-production steps; long timeline |
| Valuation Upside | ⭐⭐⭐⭐ | Significant upside if Mojave drills successfully |
| **Overall** | **⭐⭐⭐½** | **Speculative buy — high risk, high reward** |

**Bottom line:** LKY is a **high-risk, high-reward exploration play** with genuine strategic merit. 
At A$97M market cap, the stock is pricing in meaningful success at Mojave — the upcoming drill 
results will be the most significant value-determining event in the company's history. 
The copper thesis at Tottenham provides a fundamental floor; the Mojave narrative provides the 
re-rating potential. **Not appropriate for risk-averse investors. Position sizing is critical.**

---
*Disclaimer: This analysis is generated by AI and is for informational purposes only. It does not 
constitute financial advice. All valuations are estimated. Always conduct independent due diligence 
and consult a licensed financial advisor before investing.*


In [16]:
# ── Final Summary Dashboard ──────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  LOCKSLEY RESOURCES (LKY) — INVESTMENT SUMMARY                      ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Current Price:     A${CURRENT_PRICE_AUD:.3f} | Market Cap: A${MARKET_CAP_AUD_M:.0f}M          ║")
print(f"║  Live Copper:       ${LIVE_COPPER_PRICE_LB:.2f}/lb (near ATH of ${COPPER_ATH_LB:.2f}/lb)    ║".replace('ATHALB', 'ATH_LB'))
print(f"║  Cu Resource:       {TOTTENHAM_CU_CONTAINED_KT:.0f}kt Cu + {TOTTENHAM_AU_CONTAINED_KOZ:.0f}koz Au (JORC Inferred)       ║")
print( "║  Stage:             EXPLORER — pre-PEA on all assets               ║")
print( "╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  rNAV (Bear):       USD${results['bear']['rnav_ev_usdm']:>6.0f}M (= A${results['bear']['rnav_ev_usdm']/AUD_USD:>6.0f}M)                     ║")
print(f"║  rNAV (Base):       USD${results['base']['rnav_ev_usdm']:>6.0f}M (= A${results['base']['rnav_ev_usdm']/AUD_USD:>6.0f}M) [incl. Mojave: +$30M] ║")
print(f"║  rNAV (Bull):       USD${results['bull']['rnav_ev_usdm']:>6.0f}M (= A${results['bull']['rnav_ev_usdm']/AUD_USD:>6.0f}M) [incl. Mojave:+$100M] ║")
print( "╠══════════════════════════════════════════════════════════════════════╣")
print( "║  UPCOMING CATALYSTS (HIGH IMPACT):                                  ║")
print( "║  🔴 Mojave DAM & El Campo drill results (Q1 2026) — BINARY          ║")
print( "║  🟡 Tottenham updated resource estimate (Q1 2026)                   ║")
print( "║  🟢 DoD/EXIM grant announcement (2026) — non-dilutive upside        ║")
print( "╠══════════════════════════════════════════════════════════════════════╣")
print( "║  VERDICT: High-Risk / High-Reward Speculative Exploration Play       ║")
print( "║  Suitable for: Risk-tolerant growth investors (small position only)  ║")
print( "║  Not suitable for: Capital-preservation, income-focused investors    ║")
print( "╚══════════════════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════════════════╗
║  LOCKSLEY RESOURCES (LKY) — INVESTMENT SUMMARY                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  Current Price:     A$0.335 | Market Cap: A$96M          ║
║  Live Copper:       $5.90/lb (near ATH of $6.61/lb)    ║
║  Cu Resource:       71kt Cu + 70koz Au (JORC Inferred)       ║
║  Stage:             EXPLORER — pre-PEA on all assets               ║
╠══════════════════════════════════════════════════════════════════════╣
║  rNAV (Bear):       USD$    -4M (= A$    -6M)                     ║
║  rNAV (Base):       USD$    27M (= A$    43M) [incl. Mojave: +$30M] ║
║  rNAV (Bull):       USD$   111M (= A$   176M) [incl. Mojave:+$100M] ║
╠══════════════════════════════════════════════════════════════════════╣
║  UPCOMING CATALYSTS (HIGH IMPACT):                                  ║
║  🔴 Mojave DAM & El Campo drill results (Q1 2026) — BINARY          ║
║  🟡 Tottenham updated 